In [6]:
# libraries
# install once if needed: %pip install torch numpy pandas seaborn matplotlib pillow nbformat nbclient ipykernel scipy
import os
import sys
import json
import time
import subprocess
from pathlib import Path
import nbformat
import pandas as pd
from IPython.display import display

In [7]:
# globals; open this notebook with the project root as the working directory
ROOT = Path.cwd().resolve()
PROFILE = 'smoke'
DEVICE = 'auto'
RESUME = False
STAGES = ['a_import', 'b_prepare', 'c_models', 'd_phase1_train', 'e_phase1_test',
          'f_phase2_train', 'g_phase2_test', 'h_compare']
os.environ['NCA_PROJECT_ROOT'] = str(ROOT)
os.environ['NCA_PROFILE'] = PROFILE
os.environ['NCA_DEVICE'] = DEVICE
os.environ['NCA_RESUME'] = str(int(RESUME))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [8]:
# run guide; all executable analysis stays in notebooks, one pipeline entry point
guide = pd.DataFrame([
    ('start', 'extract the whole project; open this notebook from its root; run all cells'),
    ('custom images', 'optional RGBA cat.png and smiley.png in a_import/input; otherwise bundled geometric fixtures'),
    ('configuration', 'profile here; experiment settings in the globals cell of a_import/src/a_import.ipynb'),
    ('pilot', '600 Phase 1 + 900 Phase 2 optimizer steps per run; one replicate; both budget regimes'),
    ('full', '4000 + 6000 steps; three initialization/data streams; both regimes; substantial compute'),
    ('smoke', 'four steps per phase; validates execution and interfaces; no evidence of successful learning'),
    ('resume', 'set RESUME=True; same data/model/training settings; continue exact saved optimizer/pool/RNG state'),
    ('outputs', 'h_compare/output has summaries, plots, examples and animation; each upstream task retains its artifacts'),
    ('communication', 'one hard bit per directed Moore-neighbor edge per round; eight ports per interior cell'),
    ('determinism', 'synchronous updates, no dropout/fire masks; deterministic dynamics given the seed and cue schedule'),
    ('Phase 2', 'supervised conditional two-template latent generator; no VAE loss, discriminator or novel-class claim'),
    ('interpretation', 'transformation generalization, online response, repair and stochastic robustness are separate outcomes')
], columns=['item', 'detail'])
display(guide)

,item,detail
0,start,extract the whole project; open this notebook ...
1,custom images,optional RGBA cat.png and smiley.png in a_impo...
2,configuration,profile here; experiment settings in the globa...
3,pilot,600 Phase 1 + 900 Phase 2 optimizer steps per ...
4,full,4000 + 6000 steps; three initialization/data s...
5,smoke,four steps per phase; validates execution and ...
6,resume,set RESUME=True; same data/model/training sett...
7,outputs,"h_compare/output has summaries, plots, example..."
8,communication,one hard bit per directed Moore-neighbor edge ...
9,determinism,"synchronous updates, no dropout/fire masks; de..."


In [9]:
# isolated notebook executor; each stage uses a fresh Python process, no kernel sockets required
EXECUTOR_SOURCE = r'''
import sys
import traceback
from pathlib import Path
import nbformat
from IPython.terminal.interactiveshell import TerminalInteractiveShell
from IPython.utils.capture import capture_output

source, destination = map(Path, sys.argv[1:3])
notebook = nbformat.read(source, as_version=4)
shell = TerminalInteractiveShell.instance()
shell.run_line_magic('matplotlib', 'inline')
shell.display_formatter.active_types = ['text/plain', 'text/html', 'image/png', 'image/svg+xml']
for code in notebook.cells:
    with capture_output(stdout=True, stderr=True, display=True) as captured:
        result = shell.run_cell(code.source, store_history=True)
    code.execution_count = result.execution_count
    code.outputs = []
    for name in ['stdout', 'stderr']:
        if getattr(captured, name):
            code.outputs.append(nbformat.v4.new_output('stream', name=name, text=getattr(captured, name)))
    code.outputs.extend(nbformat.v4.new_output('display_data', data=out.data, metadata=out.metadata or {}) for out in captured.outputs)
    error = result.error_before_exec or result.error_in_exec
    if error is not None:
        code.outputs.append(nbformat.v4.new_output('error', ename=type(error).__name__, evalue=str(error),
                                                  traceback=traceback.format_exception(type(error), error, error.__traceback__)))
    nbformat.write(notebook, destination)
    if error is not None:
        raise RuntimeError(f'{source.name}: {error}')
'''

In [10]:
# execution; each input links only to the preceding task's output
# output handoffs are relative symlinks too; rebuilding here also repairs links after ZIP extraction
pipeline_rows = []
for index, stage in enumerate(STAGES):
    task = ROOT / stage
    for folder in ['input', 'output', 'src']:
        (task / folder).mkdir(parents=True, exist_ok=True)
    if index:
        previous = ROOT / STAGES[index - 1] / 'output'
        for source in previous.iterdir():
            if source.name in ['executed.ipynb', 'execution.log']:
                continue
            target = task / 'input' / source.name
            target.unlink(missing_ok=True)
            target.symlink_to(Path('..') / '..' / STAGES[index - 1] / 'output' / source.name,
                              target_is_directory=source.is_dir())
            handoff = task / 'output' / source.name
            handoff.unlink(missing_ok=True)
            handoff.symlink_to(Path('..') / 'input' / source.name, target_is_directory=source.is_dir())
    notebook_path = task / 'src' / f'{stage}.ipynb'
    notebook = nbformat.read(notebook_path, as_version=4)
    display(pd.DataFrame([{'stage': stage, 'profile': PROFILE, 'status': 'running'}]))
    started = time.perf_counter()
    completed = subprocess.run([sys.executable, '-c', EXECUTOR_SOURCE, str(notebook_path),
                                str(task / 'output' / 'executed.ipynb')], cwd=ROOT, capture_output=True, text=True)
    log_path = task / 'output' / 'execution.log'
    log_path.unlink(missing_ok=True)
    log_path.write_text(completed.stdout + completed.stderr)
    if completed.returncode:
        raise RuntimeError(f'{stage} failed; inspect its executed.ipynb and execution.log\n' + completed.stderr[-4000:])
    pipeline_rows.append({'stage': stage, 'profile': PROFILE, 'seconds': time.perf_counter() - started})
    display(pd.DataFrame(pipeline_rows).tail(1))
pipeline_df = pd.DataFrame(pipeline_rows)
pipeline_df.to_csv(ROOT / 'h_compare' / 'output' / 'pipeline_execution.csv', index=False)
display(pipeline_df)

,stage,profile,status
0,a_import,smoke,running


,stage,profile,seconds
0,a_import,smoke,7.016298


,stage,profile,status
0,b_prepare,smoke,running


,stage,profile,seconds
1,b_prepare,smoke,5.675549


,stage,profile,status
0,c_models,smoke,running


,stage,profile,seconds
2,c_models,smoke,3.37475


... (7 outputs left out)

,stage,profile,seconds
6,g_phase2_test,smoke,7.466822


,stage,profile,status
0,h_compare,smoke,running


,stage,profile,seconds
7,h_compare,smoke,7.892934


,stage,profile,seconds
0,a_import,smoke,7.016298
1,b_prepare,smoke,5.675549
2,c_models,smoke,3.374750
3,d_phase1_train,smoke,4.829866
4,e_phase1_test,smoke,6.263133
5,f_phase2_train,smoke,4.292731
6,g_phase2_test,smoke,7.466822
7,h_compare,smoke,7.892934


In [11]:
# final tables; plots and full cell output are in h_compare/output/executed.ipynb
display(pd.read_csv(ROOT / 'h_compare' / 'output' / 'comparison.csv'))
display(pd.read_csv(ROOT / 'h_compare' / 'output' / 'interpretation.csv'))

,phase,model,regime,split,metric,mean,n_replicates,ci_low,ci_high,profile
0,1,1A_bern,bits,test_angle,mse,0.032357,1,NaN,NaN,smoke
1,1,1A_bern,bits,test_angle,foreground_rmse,0.567691,1,NaN,NaN,smoke
2,1,1A_bern,bits,test_angle,alpha_iou,0.000000,1,NaN,NaN,smoke
3,1,1A_bern,bits,test_color,mse,0.034568,1,NaN,NaN,smoke
4,1,1A_bern,bits,test_color,foreground_rmse,0.596728,1,NaN,NaN,smoke
...,...,...,...,...,...,...,...,...,...,...
355,2,1B_rbern,updates,test_joint,foreground_rmse,0.607400,1,NaN,NaN,smoke
356,2,1B_rbern,updates,test_joint,alpha_iou,0.000000,1,NaN,NaN,smoke
357,2,1B_rbern,updates,test_translation,mse,0.041764,1,NaN,NaN,smoke
358,2,1B_rbern,updates,test_translation,foreground_rmse,0.644914,1,NaN,NaN,smoke


,item,interpretation
0,run status,SMOKE: execution check only
1,evidence threshold,smoke validates execution only; pilot is explo...
2,uncertainty,replicate bootstrap across trained runs; one-r...
3,calibration uncertainty,latent-level bootstrap within run; repeated co...
4,fairness,both equal updates and equal bits are trained;...
5,pose/color,conditional transformation generalization; the...
6,seed access,only the seed receives continuous environmenta...
7,bandwidth,"one bit per directed edge per round, not one t..."
8,deterministic models,fixed seed/cues produce one trajectory; differ...
9,stochastic models,Bernoulli probability floor prevents arbitrari...
